# 02 — Features

Builds team-week EPA stats with rolling windows, then assembles the game-level feature table.

Key principle: **all features are shifted by 1 game** — a team's stats for week N use only data through week N-1, so we never leak the future into the past.

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data/raw')
OUT_DIR = Path('../data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

pbp = pl.read_parquet(DATA_DIR / 'pbp.parquet')
schedules = pl.read_parquet(DATA_DIR / 'schedules.parquet')
print('PBP:', pbp.shape, '| Schedules:', schedules.shape)

## Team-week EPA stats (offense + defense)

In [ ]:
# Only real offensive plays — drop kneels, spikes, special teams, penalties-only
plays = pbp.filter(
    pl.col('play_type').is_in(['pass', 'run']) &
    pl.col('epa').is_not_null() &
    (pl.col('season_type') == 'REG')
)

off_stats = (
    plays.group_by(['season', 'week', 'posteam'])
    .agg([
        pl.col('epa').mean().alias('off_epa_play'),
        pl.col('success').mean().alias('off_success_rate'),
        (pl.col('epa') > 0.5).mean().alias('off_explosive_rate'),
        pl.col('epa').count().alias('off_plays'),
    ])
    .rename({'posteam': 'team'})
)

def_stats = (
    plays.group_by(['season', 'week', 'defteam'])
    .agg([
        pl.col('epa').mean().alias('def_epa_play'),
        pl.col('success').mean().alias('def_success_rate'),
        (pl.col('epa') > 0.5).mean().alias('def_explosive_rate'),
    ])
    .rename({'defteam': 'team'})
)

team_week = off_stats.join(def_stats, on=['season','week','team'], how='inner').to_pandas()
team_week = team_week.sort_values(['team','season','week']).reset_index(drop=True)
print('Team-week rows:', len(team_week))
team_week.head()

## Rolling windows (lagged by 1 — no leakage)

In [ ]:
stat_cols = ['off_epa_play','off_success_rate','off_explosive_rate',
             'def_epa_play','def_success_rate','def_explosive_rate']

tw = team_week.copy()
for c in stat_cols:
    # last-4-game and last-8-game rolling avg, lagged by 1
    tw[f'{c}_l4'] = tw.groupby('team')[c].transform(
        lambda s: s.shift(1).rolling(4, min_periods=1).mean()
    )
    tw[f'{c}_l8'] = tw.groupby('team')[c].transform(
        lambda s: s.shift(1).rolling(8, min_periods=1).mean()
    )
    # season-to-date, lagged by 1
    tw[f'{c}_ytd'] = tw.groupby(['team','season'])[c].transform(
        lambda s: s.shift(1).expanding().mean()
    )

tw.to_parquet(OUT_DIR / 'team_week.parquet')
print('Saved team_week with rolling features')
tw.filter(items=['team','season','week','off_epa_play','off_epa_play_l4','off_epa_play_ytd']).head(10)

## Game-level feature table

In [ ]:
sched = schedules.to_pandas()
games = sched[(sched['game_type'] == 'REG') & sched['result'].notna()].copy()
# nflverse convention: result = home_score - away_score (positive = home win)
games['margin'] = games['home_score'] - games['away_score']
print('Completed regular-season games:', len(games))

In [ ]:
# Build separate home/away feature tables and merge
feat_cols = [c for c in tw.columns if c not in ['season','week','team']]

home_feats = tw.rename(columns={'team':'home_team', **{c: f'home_{c}' for c in feat_cols}})
away_feats = tw.rename(columns={'team':'away_team', **{c: f'away_{c}' for c in feat_cols}})

g = games.merge(home_feats, on=['season','week','home_team'], how='left')
g = g.merge(away_feats, on=['season','week','away_team'], how='left')

# Differential features (home - away) — these are usually the strongest signal
for c in stat_cols:
    for w in ['l4','l8','ytd']:
        g[f'diff_{c}_{w}'] = g[f'home_{c}_{w}'] - g[f'away_{c}_{w}']

g['rest_diff'] = g['home_rest'] - g['away_rest']

g.to_parquet(OUT_DIR / 'game_features.parquet')
print(f'Saved {len(g)} games with features')
g[['game_id','season','week','home_team','away_team','margin','spread_line','total_line','diff_off_epa_play_l4']].head(10)

## Done
Next: open `03_train_backtest.ipynb` to fit Elo + XGBoost and run the walk-forward backtest.